# Paisabazaar Banking Fraud Analysis
## Credit Score Classification and Exploratory Data Analysis

**Important scope note:** The provided target is `Credit_Score` with classes Good, Standard, and Poor. This dataset supports credit-score classification, not fraud detection.

### Objectives
- Explore 100,000 customer-month records and assess data quality.
- Answer the brief’s questions about income, utilization/loans, card count, delayed payments, age groups, and month trends.
- Fit a baseline model with a customer-level holdout to reduce leakage from repeated monthly records.

### Run in Google Colab
Run the next cell, select `dataset (1).csv` when prompted, then choose **Runtime → Run all**.


In [ ]:
# Google Colab file picker: choose dataset (1).csv from your computer
from google.colab import files
uploaded = files.upload()
DATA_FILE = next(iter(uploaded))
print("Loaded file:", DATA_FILE)


In [ ]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
sns.set_theme(style="whitegrid", palette="Set2")

df = pd.read_csv(DATA_FILE, low_memory=False)
df.columns = df.columns.str.strip()
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())


## 1. Data overview and quality checks
The file contains repeated monthly observations: each of the 12,500 customers has 8 rows. We will keep customer groups together in the model split.


In [ ]:
print("Rows:", f"{len(df):,}", " | Columns:", len(df.columns))
print("Unique customers:", f"{df['Customer_ID'].nunique():,}")
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Missing cells:", int(df.isna().sum().sum()))
print("Rows per customer (distribution):")
display(df.groupby("Customer_ID").size().value_counts().sort_index().rename_axis("rows_per_customer").to_frame("customers"))
quality = pd.DataFrame({"dtype": df.dtypes.astype(str), "missing_count": df.isna().sum(), "missing_pct": (100*df.isna().mean()).round(2), "unique_values": df.nunique(dropna=False)})
display(quality)


## 2. Target distribution and credit score across age groups


In [ ]:
score_order = ["Poor", "Standard", "Good"]
score_counts = df["Credit_Score"].value_counts().reindex(score_order)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x="Credit_Score", order=score_order, ax=ax[0])
ax[0].set(title="Number of records by credit score", xlabel="Credit score", ylabel="Records")
score_pct = (score_counts/len(df)*100).round(1)
ax[1].bar(score_pct.index, score_pct.values, color=sns.color_palette("Set2", 3))
ax[1].set(title="Credit score share", xlabel="Credit score", ylabel="Percent of records")
for i,v in enumerate(score_pct.values): ax[1].text(i,v+0.5,f"{v}%",ha="center")
plt.tight_layout(); plt.show()

# Age is repeated monthly per customer; grouped age bands summarize the portfolio profile.
age = pd.to_numeric(df["Age"], errors="coerce")
df["Age_Group"] = pd.cut(age, bins=[0,25,35,45,55,65,120], labels=["18–25","26–35","36–45","46–55","56–65","66+"])
age_share = pd.crosstab(df["Age_Group"], df["Credit_Score"], normalize="index").reindex(columns=score_order)
ax=age_share.plot(kind="bar", stacked=True, figsize=(10,5), color=sns.color_palette("Set2",3))
ax.set(title="Credit score mix across age groups", xlabel="Age group", ylabel="Share within age group"); ax.legend(title="Credit score", bbox_to_anchor=(1.02,1)); plt.xticks(rotation=0); plt.tight_layout(); plt.show()


## 3. Income, utilization, loans, cards, and delayed payments
Numeric-looking fields are converted safely for charts. We cap only the displayed annual-income axis at the 99th percentile to make the central distribution readable; raw data is retained.


In [ ]:
def to_num(s):
    return pd.to_numeric(s.astype("string").str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce")
for col in ["Annual_Income","Credit_Utilization_Ratio","Num_Credit_Card","Num_of_Loan","Delay_from_due_date","Num_of_Delayed_Payment"]:
    if col in df: df[col+"_clean"] = to_num(df[col])

# Income distribution by credit score (box plot)
p=df[["Annual_Income_clean","Credit_Score"]].dropna().copy(); cap=p.Annual_Income_clean.quantile(.99); p["income_display"]=p.Annual_Income_clean.clip(upper=cap)
plt.figure(figsize=(9,5)); sns.boxplot(data=p,x="Credit_Score",y="income_display",order=score_order,showfliers=False)
plt.title("Annual income by credit score (display capped at 99th percentile)"); plt.xlabel("Credit score"); plt.ylabel("Annual income"); plt.tight_layout(); plt.show()

# Utilization across loan count bands; observations, not causal effects
t=df[["Credit_Utilization_Ratio_clean","Num_of_Loan_clean","Credit_Score"]].dropna().copy()
t["Loan_Band"]=pd.cut(t.Num_of_Loan_clean,[-1,0,2,5,100],labels=["0","1–2","3–5","6+"])
plt.figure(figsize=(10,5)); sns.barplot(data=t,x="Loan_Band",y="Credit_Utilization_Ratio_clean",hue="Credit_Score",hue_order=score_order,errorbar=("ci",95))
plt.title("Credit utilization by loan-count band and credit score"); plt.xlabel("Number of loans"); plt.ylabel("Mean utilization ratio"); plt.tight_layout(); plt.show()

# Credit card count by score (stacked proportions)
t=df[["Num_Credit_Card_clean","Credit_Score"]].dropna().copy(); t["Card_Band"]=pd.cut(t.Num_Credit_Card_clean,[-1,1,3,5,100],labels=["0–1","2–3","4–5","6+"])
card_share=pd.crosstab(t.Card_Band,t.Credit_Score,normalize="index").reindex(columns=score_order)
ax=card_share.plot(kind="bar",stacked=True,figsize=(9,5),color=sns.color_palette("Set2",3)); ax.set(title="Credit score mix by number of credit cards",xlabel="Credit card count",ylabel="Share within band"); ax.legend(title="Credit score"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

# Delayed payment count and due-date delay
fig,ax=plt.subplots(1,2,figsize=(13,5))
sns.boxplot(data=df,x="Credit_Score",y="Num_of_Delayed_Payment_clean",order=score_order,showfliers=False,ax=ax[0]); ax[0].set(title="Delayed payment count",xlabel="Credit score",ylabel="Count")
sns.boxplot(data=df,x="Credit_Score",y="Delay_from_due_date_clean",order=score_order,showfliers=False,ax=ax[1]); ax[1].set(title="Days delayed from due date",xlabel="Credit score",ylabel="Days")
plt.tight_layout(); plt.show()


## 4. Month trends and numeric relationships


In [ ]:
month_score = pd.crosstab(df["Month"],df["Credit_Score"],normalize="index").reindex(columns=score_order)
ax=month_score.plot(marker="o",figsize=(10,5)); ax.set(title="Credit score mix by month",xlabel="Month number",ylabel="Share of records"); ax.set_xticks(sorted(df.Month.unique())); plt.tight_layout(); plt.show()

num_cols=["Annual_Income","Num_Bank_Accounts","Num_Credit_Card","Interest_Rate","Num_of_Loan","Delay_from_due_date","Num_of_Delayed_Payment","Num_Credit_Inquiries","Outstanding_Debt","Credit_Utilization_Ratio","Credit_History_Age","Total_EMI_per_month","Monthly_Balance"]
num_cols=[c for c in num_cols if c in df]
plt.figure(figsize=(12,9)); sns.heatmap(df[num_cols].corr(),cmap="vlag",center=0,annot=True,fmt=".2f",annot_kws={"size":7}); plt.title("Correlation among numeric variables"); plt.tight_layout(); plt.show()

# Compact class comparison for key variables
key=["Annual_Income","Num_Credit_Card","Delay_from_due_date","Num_of_Delayed_Payment","Credit_History_Age","Outstanding_Debt","Credit_Utilization_Ratio"]
display(df.groupby("Credit_Score")[key].median().reindex(score_order).round(2))


## 5. Baseline credit score classifier
A random 80/20 row split would place monthly records from the same customer on both sides and inflate performance. This baseline splits by `Customer_ID`. Direct identifiers (`ID`, `Customer_ID`, `Name`, `SSN`) and the high-cardinality loan-description field are excluded. Compare macro metrics and per-class recall; do not interpret this baseline as lending approval advice.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, balanced_accuracy_score, ConfusionMatrixDisplay

target="Credit_Score"
drop_cols=[c for c in [target,"ID","Customer_ID","Name","SSN","Type_of_Loan","Age_Group"] if c in df]
X=df.drop(columns=drop_cols).copy(); y=df[target].astype(str)
# Convert numeric-looking text fields, leaving genuine categories for one-hot encoding.
for c in X.select_dtypes(include=["object","string"]).columns:
    converted=to_num(X[c])
    if converted.notna().mean()>.85: X[c]=converted
num_features=X.select_dtypes(include=np.number).columns.tolist(); cat_features=[c for c in X if c not in num_features]
preprocess=ColumnTransformer([
 ("numeric",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num_features),
 ("categorical",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("encode",OneHotEncoder(handle_unknown="ignore",min_frequency=10))]),cat_features)
])
model=Pipeline([("preprocess",preprocess),("classifier",RandomForestClassifier(n_estimators=250,min_samples_leaf=2,class_weight="balanced_subsample",random_state=42,n_jobs=-1))])
split=GroupShuffleSplit(n_splits=1,test_size=.20,random_state=42)
train_idx,test_idx=next(split.split(X,y,groups=df["Customer_ID"]))
model.fit(X.iloc[train_idx],y.iloc[train_idx]); pred=model.predict(X.iloc[test_idx])
print("Customer-grouped holdout balanced accuracy:",round(balanced_accuracy_score(y.iloc[test_idx],pred),3))
print(classification_report(y.iloc[test_idx],pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y.iloc[test_idx],pred,labels=score_order,display_labels=score_order,normalize="true",cmap="Blues",xticks_rotation=20)
plt.title("Normalized confusion matrix — customer-grouped holdout"); plt.tight_layout(); plt.show()


## 6. Findings, limitations, and stakeholder value

### Findings observed in this dataset
- 100,000 records represent 12,500 customers over 8 months each. There are 28 columns, no exact duplicate rows, and no null cells as read by pandas.
- Credit score distribution is imbalanced: Standard 53.2%, Poor 29.0%, Good 17.8%. Class-specific metrics are needed.
- Median annual income is about 44,928 for Good, 36,824 for Standard, and 32,065 for Poor records. This is an association, not evidence that income causes a score.
- Median days delayed are 10 (Good), 18 (Standard), and 27 (Poor); delayed payment count medians are 8, 14, and 17. Payment behavior is strongly differentiated across score classes in this sample.
- Median credit history age is 288 months (Good), 227 (Standard), and 161 (Poor). Poor-score records also have higher median card count (7 versus 4 for Good) and outstanding debt (about 2,081 versus 801).
- Median utilization is similar across classes (about 32%); in this dataset it separates score classes less than payment delays or credit history age.
- Standard remains the largest class in every month. Good-score share rises from about 15.8% in months 1–3 to about 19.3% in months 7–8; the time span is only eight months, so this pattern should not be treated as a stable seasonal effect.

### Data and modeling limitations
- The eight records per customer are repeated observations; group splitting reduces but does not replace a true forward-in-time evaluation.
- Several fields may be derived from the same credit bureau process that produced the target. Confirm feature timing and provenance before deployment.
- Check malformed values, outliers, subgroup performance, calibration, and class-specific costs. No causal conclusions are supported by this observational analysis.
- Dataset documentation describes U.S. identifiers such as SSN; direct personal identifiers are excluded from the model.

### Stakeholder value
Paisabazaar can use this analysis to understand which customer attributes co-occur with credit score classes, prioritize data quality improvements, and support more relevant financial guidance. Any model should assist review, not make unvalidated or adverse credit decisions on its own.
